In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import zipfile

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
file='/content/drive/MyDrive/TransformersCode/e_commerce/chatbot/estore_qa.zip'
with zipfile.ZipFile(file,'r') as zip_ref:
  zip_ref.extractall('/content/drive/MyDrive/TransformersCode/e_commerce/chatbot')

df=pd.read_csv('/content/drive/MyDrive/TransformersCode/e_commerce/chatbot/estore_qa.csv')
df

,question,answer
0,What are the payment options available?,"We accept credit/debit cards, PayPal, and bank..."
1,Do you offer international shipping?,"Yes, we ship to many countries worldwide. Chec..."
2,What is the return policy?,You can return items within 30 days of receipt...
3,How can I track my order?,You can track your order using the tracking nu...
4,What should I do if I receive a damaged item?,Please contact our customer support for assist...
...,...,...
95,How do I redeem a promotional offer?,Enter the promotional code at checkout to rede...
96,What should I do if I receive an incorrect cha...,Contact support if you notice an incorrect cha...
97,Can I request a product demo?,Request a product demo by contacting our sales...
98,How do I submit a warranty claim?,Submit a warranty claim through the 'Warranty'...


In [5]:
from datasets import Dataset
from transformers import AutoTokenizer

# دمج السؤال والجواب في نص واحد لكل صف
df['text'] = "Question: " + df['question'] + "\nAnswer: " + df['answer'] + "<|endoftext|>"

# تحويل الـ DataFrame إلى Dataset الخاصة بـ Hugging Face
dataset = Dataset.from_pandas(df[['text']])

# تحميل المرمز (Tokenizer)
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# دالة المعالجة
def tokenize_function(examples):
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [6]:
from datasets import Dataset
from transformers import GPT2LMHeadModel, GPT2Tokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import pandas as pd
import torch

# 1. التأكد من الكارت أو المعالج
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. التأكد من تجهيز النصوص (افترضنا أن لديك DataFrame باسم df يحتوي على question و answer)
# إذا كانت مجهزة مسبقاً، يمكنك تخطي هذا السطر
if 'text' not in df.columns:
    df['text'] = "Question: " + df['question'] + "\nAnswer: " + df['answer'] + "<|endoftext|>"

# تحويل الـ DataFrame إلى Dataset
dataset = Dataset.from_pandas(df[['text']])

# 3. تحميل المرمز (Tokenizer) والنموذج
model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)
model.to(device)

# 4. دالة المعالجة وتطبيقها لإنشاء tokenized_dataset
def tokenize_function(examples):
    tokens = tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# 5. إعداد DataCollator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# 6. إعدادات التدريب
training_args = TrainingArguments(
    output_dir="./gpt2-faq-bot",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    save_strategy="no",
    logging_steps=5,
    learning_rate=5e-5,
    fp16=True if device == "cuda" else False, # تفادي خطأ fp16 إذا كان يعمل على CPU
)

# 7. إعداد المدرب (Trainer)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

# 8. بدء التدريب الفوري
print("🚀 يبدأ التدريب الآن...")
trainer.train()

# 9. الحفظ
model.save_pretrained("./gpt2-faq-bot")
tokenizer.save_pretrained("./gpt2-faq-bot")
print("✅ تم الانتهاء من التدريب والحفظ بنجاح!")

Using device: cpu


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

🚀 يبدأ التدريب الآن...


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,2.199639
10,1.551127
15,1.246156
20,1.128383
25,1.192069
30,0.963566
35,0.947170
40,0.950679
45,0.867658
50,0.813455


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ تم الانتهاء من التدريب والحفظ بنجاح!


In [7]:
trainer.save_model("./gpt2-faq-bot")
tokenizer.save_pretrained("./gpt2-faq-bot")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./gpt2-faq-bot/tokenizer_config.json', './gpt2-faq-bot/tokenizer.json')